In [1]:
#
import requests
import pandas as pd 
pd.set_option('display.max_rows', None)
from IPython.display import HTML

#
import mysql.connector
from mysql.connector import Error

#
import numpy as np

#
import os 
from dotenv import load_dotenv
load_dotenv()
password_sql = os.getenv("PASS_SQL")

#
import json

# EXTRACCIÓN DE DATOS EN DEEZER Y LAST.FM

In [2]:
# Función genérica para extraer las API 
def extraccion_API(endpoint, verbose=True):

    try:
        # Endpoint como argumento de función
        datos = requests.get(endpoint)
        if datos.status_code == 200:
            # Se permite print detallado
            if verbose: 
                print ("API conectada correctamente")
            # Se convierte datos a JSON para que sea manejable por el resto de funciones
            datos_json = datos.json()
            return datos_json
        else:
            # En caso de error, se imprime fallo 
            print (f"Error de conexión en API {endpoint}: {datos.status_code}")

    # Error de conexión        
    except requests.exceptions.ConnectionError as CnxE:
        print (CnxE)

    # Error de tiempo de espera
    except requests.exceptions.Timeout as TO:
        print (TO)
    
    # Cubrimos el resto de errores
    except requests.exceptions.RequestException as e:
        print (e) 

In [3]:
# Lista los ID de los cantantes o grupos que se van a extraer
id_artistas = [12246, 160, 145, 564, 75491, 75798, 290, 483, 10803980, 1538640, 892, 10583405, 412, 13, 4050205, 384236, 119, 5620251, 259, 5962948, 196, 485, 425, 315929, 2446, 1755, 180, 98, 10977, 1434]

In [5]:
# Función para extraer los datos de artistas de Deezer
def extraer_artistas(id_artistas):

    # Se define lista vacía y el contador de verbose
    lista_artistas = []
    contador = 0
    # Se recorre un endpoint por artista
    for artistas in id_artistas:
        endpoint = f"https://api.deezer.com/artist/{artistas}"

        # Se lanza la función para extraer las API
        datos_artistas = extraccion_API(endpoint, verbose=False)
        contador += 1
        # Diccionario con los datos necesarios
        diccionario_artista = {
            "id_artista": datos_artistas["id"],
            "nombre": datos_artistas["name"]
            }
        lista_artistas.append(diccionario_artista)

    # Imprime el número de apis correctas
    print(f"{contador} APIs extraídas correctamente")
    # Unimos la lista de DF
    df_final = pd.DataFrame(lista_artistas)
    return df_final

In [6]:
df_artistas = extraer_artistas(id_artistas)
HTML(df_artistas.to_html(index=False))

30 APIs extraídas correctamente


id_artista,nombre
12246,Taylor Swift
160,Shakira
145,Beyoncé
564,Rihanna
75491,Lady Gaga
75798,Adele
290,Madonna
483,Britney Spears
10803980,BLACKPINK
1538640,Little Mix


In [7]:
# Función para extraer los datos de las canciones de Deezer
def extraer_canciones(id_artistas):

    # Lista vacía para el DF y contador
    lista_canciones = []
    contador = 0
    # Se recorre un endpoint por artista
    for artistas in id_artistas:
        endpoint = f"https://api.deezer.com/artist/{artistas}/top?limit=50"
        
        # Se lanza la función para extraer las API
        datos_canciones = extraccion_API(endpoint, verbose = False)
        contador += 1

        # Bucle para recoger los datos que necesitamos del JSON
        for cancion in datos_canciones["data"]:
            id_cancion = cancion["id"]
            id_artista = cancion["artist"]["id"]
            id_album = cancion["album"]["id"]
            titulo = cancion["title"]
            duracion = cancion["duration"]
            ranking = cancion["rank"]
            if len(cancion["contributors"]) == 1:
                colaboradores = False
            else:
                colaboradores = True

            # Diccionario creado con los datos extraidos del JSON
            diccionario_cancion = {
                    "id_cancion": id_cancion,
                    "id_artista": id_artista,
                    "id_album": id_album,
                    "title": titulo,
                    "duration": duracion,
                    "rank": ranking,
                    "contributors_bool": colaboradores
                }         
                        
            # Se añade el diccionario a la lista
            lista_canciones.append(diccionario_cancion)
    
    # Imprime el número de apis correctas
    print(f"{contador} APIs extraídas correctamente")        
    df_canciones = pd.DataFrame(lista_canciones)
    df_final = df_canciones.drop_duplicates(subset="id_cancion")
    return df_final, lista_canciones

In [8]:
df_canciones ,  lista_canciones = extraer_canciones(id_artistas) 
df_canciones.to_csv("listado_de_canciones.csv", index=False)

30 APIs extraídas correctamente


In [9]:
# Funcion para comprobar que los artistas elegidos tienen 50 canciones
def conteo_artistas(id_artistas):

    # Diccionario vacío para mostrar conteo
    diccionario_artistas = {}
    # Se recorren las APIs
    for artistas in id_artistas:
        endpoint = f"https://api.deezer.com/artist/{artistas}/top?limit=50"
        datos = extraccion_API(endpoint, verbose=False)
        
        # Enfrenta artistas con el número de canciones en la API
        diccionario_artistas[artistas] = len(datos["data"])
    return diccionario_artistas

In [10]:
diccionario_artistas = conteo_artistas(id_artistas)
diccionario_artistas

{12246: 50,
 160: 50,
 145: 50,
 564: 50,
 75491: 50,
 75798: 50,
 290: 50,
 483: 50,
 10803980: 50,
 1538640: 50,
 892: 50,
 10583405: 50,
 412: 50,
 13: 50,
 4050205: 50,
 384236: 50,
 119: 50,
 5620251: 50,
 259: 50,
 5962948: 50,
 196: 50,
 485: 50,
 425: 50,
 315929: 50,
 2446: 50,
 1755: 50,
 180: 50,
 98: 50,
 10977: 50,
 1434: 50}

In [11]:
# Función para extraer los géneros de Deezer
def extraer_genero():

    # Endpoint único
    endpoint = "https://api.deezer.com/genre/"
    try:
        datos_genero = extraccion_API(endpoint, verbose = True)
        df_genero = pd.DataFrame(datos_genero["data"])
        df_final = df_genero[["id","name"]]                       
        return df_final
    except:
        print ("error")

In [12]:
df_genero = extraer_genero()
HTML(df_genero.to_html(index=False))

API conectada correctamente


id,name
0,Todos
132,Pop
116,Rap/Hip Hop
122,Reggaeton
152,Rock
113,Dance
165,R&B
85,Alternativo
106,Electro
466,Folk


In [13]:
def extraer_album():
    album_ids_unicos = set()

    for cancion in lista_canciones:
        album_ids_unicos.add(cancion["id_album"])

    contador = 0
    lista_albumes = []
    for album_id in album_ids_unicos:
        endpoint = f"https://api.deezer.com/album/{album_id}"
        datos_albumes = extraccion_API(endpoint, verbose = False)
        contador += 1
        
        diccionario_albumes = {
            "id_album": datos_albumes["id"],
            "id_artista": datos_albumes["artist"]["id"],
            "id_genre": datos_albumes["genre_id"],
            "titulo": datos_albumes["title"],
            "n_canciones": datos_albumes["nb_tracks"],
            "fecha_lanzamiento": datos_albumes["release_date"]
        }     

        
        lista_albumes.append(diccionario_albumes)
    
    print(f"{contador} APIs extraídas correctamente") 
    df_final = pd.DataFrame(lista_albumes)
    return df_final
        


In [14]:
df_albumes = extraer_album()

603 APIs extraídas correctamente


In [15]:
def ajuste_genero():
    generos_faltantes = set(df_albumes["id_genre"]) - set(df_genero["id"])
    lista_generos_nuevos = []
    for genero in generos_faltantes:
        endpoint = f"https://api.deezer.com/genre/{genero}"
        generos_extraidos = extraccion_API(endpoint, verbose=True)
        if "error" in generos_extraidos:
            df_albumes.loc[df_albumes["id_genre"] == genero, "id_genre"] = 0
        else:    
            lista_generos_nuevos.append(generos_extraidos)
    
    if len(lista_generos_nuevos) > 0:
        df_final = pd.DataFrame(lista_generos_nuevos)[["id", "name"]]
        return df_final


In [16]:
df_albumes.to_csv("listado_de_albumes.csv", index=False)
df_ajuste_genero = ajuste_genero()
df_genero_final = pd.concat([df_genero, df_ajuste_genero], ignore_index=True)
HTML(df_genero_final.to_html(index=False))

API conectada correctamente
API conectada correctamente


id,name
0,Todos
132,Pop
116,Rap/Hip Hop
122,Reggaeton
152,Rock
113,Dance
165,R&B
85,Alternativo
106,Electro
466,Folk


In [17]:
def extraer_last_fm(df_artistas):

    lista_artistas = []
    contador = 0
    df_artistas["nombre"].str.replace(" ", "+")
    for nombre in df_artistas["nombre"]:
        endpoint = f"https://ws.audioscrobbler.com/2.0/?method=artist.getInfo&artist={nombre}&api_key=68eddfdc6b072ad56527a4199d979038&format=json"
        datos_last_fm = extraccion_API(endpoint, verbose=False)
        contador += 1
    
        diccionario_artistas = {
            "nombre" : datos_last_fm["artist"]["name"],
            "oyentes" : datos_last_fm["artist"]["stats"]["listeners"],
            "reproducciones" : datos_last_fm["artist"]["stats"]["playcount"],
            "biografia" : datos_last_fm["artist"]["bio"]["summary"]
        }
            
        
        lista_artistas.append(diccionario_artistas)

    df_final = pd.DataFrame(lista_artistas)
    df_final["biografia"] = df_final["biografia"].str.replace("\n", " ")
    print(f"{contador} APIs extraídas correctamente") 
    return df_final
        


In [18]:
df_artistas_fm = extraer_last_fm(df_artistas)
df_artistas_fm.to_csv("listado_de_artistas_fm.csv", index=False)

30 APIs extraídas correctamente


In [43]:
id_colaboradores = (set(df_albumes["id_artista"]) | set(df_canciones["id_artista"])) - set(id_artistas)
id_colaboradores

{11,
 230,
 359,
 413,
 556,
 1446,
 2896,
 3098,
 4088,
 4347,
 4474,
 4479,
 4962,
 5080,
 5828,
 7343,
 8631,
 9219,
 12178,
 75332,
 127322,
 165930,
 210977,
 246791,
 310260,
 380955,
 407188,
 554792,
 712271,
 1020109,
 1562681,
 1672366,
 3265001,
 3922661,
 3968561,
 4104927,
 4331004,
 4390053,
 4495513,
 4649104,
 4860761,
 4968870,
 5297021,
 5531258,
 5835993,
 5904266,
 6396188,
 6397900,
 7072729,
 7201102,
 7358224,
 7457468,
 7543848,
 7961888,
 8376040,
 8706544,
 9236850,
 9759672,
 9761322,
 9999412,
 11289472,
 12170972,
 12382106,
 12487862,
 58568762,
 67972932,
 79181242,
 98840272,
 108420982,
 141992442,
 207559207,
 213208547}

In [44]:
df_colaboradores = extraer_artistas(id_colaboradores)

72 APIs extraídas correctamente


In [45]:
df_colaboradores_fm = extraer_last_fm(df_colaboradores)

72 APIs extraídas correctamente


In [87]:
df_artistas["nombre"] = df_artistas["nombre"].str.lower()
df_artistas_fm["nombre"] = df_artistas_fm["nombre"].str.lower()
df_colaboradores["nombre"] = df_colaboradores["nombre"].str.lower()
df_colaboradores_fm["nombre"] = df_colaboradores_fm["nombre"].str.lower()
df_combinado_principales= pd.merge(df_artistas, df_artistas_fm, left_on="nombre", right_on="nombre")
df_combinado_colaboradores = pd.merge(df_colaboradores, df_colaboradores_fm, left_on="nombre", right_on="nombre")
df_combinado_artistas = pd.concat([df_combinado_principales, df_combinado_colaboradores], ignore_index=True)
df_combinado_artistas = df_combinado_artistas.drop_duplicates(subset="id_artista")
df_combinado_artistas

,id_artista,nombre,oyentes,reproducciones,biografia
0,12246,taylor swift,5989745,3716657384,Taylor Alison Swift is an American singer-song...
1,160,shakira,4939814,183414959,Shakira Isabel Mebarak Ripoll is a Colombian s...
2,145,beyoncé,6434581,708060557,Beyoncé Giselle Knowles-Carter (born Septembe...
3,564,rihanna,8359180,612131922,"Robyn Rihanna Fenty (born February 20, 1988), ..."
4,75491,lady gaga,7780825,1047223704,Stefani Joanne Angelina Germanotta (born 28 Ma...
5,75798,adele,5665588,321459407,"Adele Laurie Blue Adkins MBE (born May 5, 1988..."
6,290,madonna,5716862,372167910,"Madonna Louise Ciccone (born August 16, 1958) ..."
7,483,britney spears,6511637,501505752,"Britney Jean Spears (born December 2, 1981 in..."
8,10803980,blackpink,2032427,318339888,BLACKPINK (Hangul: 블랙핑크; Katakana :ブラックピンク; st...
9,1538640,little mix,1871879,114871114,Little Mix is a British girl group formed in 2...


# CREACION DE BASE DE DATOS

In [23]:
def conectar_mysql(host="127.0.0.1", user="root", password=password_sql, database=None):
    try:
        cnx = mysql.connector.connect(
            host=host,
            user=user,
            password=password,
            database=database        # si no le pasas base de datos, se conecta al servidor solo
        )
        print("Conexión exitosa")
        return cnx                      # con este return guarda la conexion y puedo usar esta conexion despues 
    except Error as e:
        print(f"Error al conectar: {e}")

In [24]:
conexion = conectar_mysql()

Conexión exitosa


In [49]:
nombre_bd = "proyecto_music_stream_team1"

In [50]:
# 
def crear_basededatos(nombre_bd):
   
    try:
        # 
        with conexion.cursor() as cursor:
            query = f"CREATE DATABASE IF NOT EXISTS {nombre_bd}"
            # 
            cursor.execute(query)
            print ("Query exitosa")
 
    # 
    except Error as e:
        print (f"Error creando base de datos: {e}")

In [89]:
crear_basededatos(nombre_bd)

Query exitosa


In [47]:
def borrar_base_datos(db_name):
    cursor = None
    try:
        with conexion.cursor() as cursor:
            cursor.execute(f"DROP DATABASE IF EXISTS {db_name}")
            print(f"Base de datos {db_name} eliminada correctamente")
    except Error as e:
        print(f"Error al eliminar la base de datos: {e}")

In [88]:
borrar_base_datos(nombre_bd)

Base de datos proyecto_music_stream_team1 eliminada correctamente


In [52]:
# para crear tablas
def crear_tablas_genericas(nombre_bd, nombre_tabla, tabla_esquema):
   
    try:
        # 
        with conexion.cursor() as cursor:
            cursor.execute(f"USE {nombre_bd};")
            # 
            query = f''' CREATE TABLE IF NOT EXISTS {nombre_tabla} ({tabla_esquema});'''
            #
            cursor.execute(query)
            print ("Query creación exitosa")
   
    #
    except Error as e:
        print (f"Error creando tabla: {e}")

In [53]:
tabla_artista = 'artista'
tabla_genero_musical = 'genero_musical'
tabla_canciones = 'canciones'
tabla_album = 'album'

In [54]:
esquema_artista = '''id_artista INT PRIMARY KEY,
   nombre VARCHAR(30) NOT NULL,
   oyentes INT,
   reproducciones BIGINT,
   biografia VARCHAR(1000) NOT NULL,
   genero VARCHAR(10),
   artista_principal BOOLEAN
   '''

In [55]:
esquema_genero_musical = '''id_genero INT PRIMARY KEY, 
nombre VARCHAR(40)'''

In [57]:
esquema_canciones = '''id_cancion BIGINT PRIMARY KEY,
id_artista INT NOT NULL,
id_album BIGINT NOT NULL,
titulo VARCHAR(200) NOT NULL,
duracion INT,
ranking_lista INT,
colaboraciones BOOLEAN,
FOREIGN KEY (id_artista)
   REFERENCES artista(id_artista),
FOREIGN KEY (id_album)
   REFERENCES album(id_album)
'''

In [58]:
esquema_album = ''' id_album BIGINT PRIMARY KEY,
id_artista INT,
id_genero INT,
titulo VARCHAR(200) NOT NULL,
numero_canciones INT,
fecha_lanzamiento DATE,
FOREIGN KEY (id_genero)
   REFERENCES genero_musical(id_genero),
FOREIGN KEY (id_artista)
   REFERENCES artista(id_artista)
'''

In [90]:
crear_tablas_genericas(nombre_bd, tabla_genero_musical, esquema_genero_musical)

Query creación exitosa


In [91]:
crear_tablas_genericas(nombre_bd,tabla_artista, esquema_artista)

Query creación exitosa


In [92]:
crear_tablas_genericas(nombre_bd,tabla_album, esquema_album)

Query creación exitosa


In [93]:
crear_tablas_genericas(nombre_bd,tabla_canciones, esquema_canciones)

Query creación exitosa


# INSERCIÓN EN MYSQL

In [94]:
cursor = conexion.cursor()

In [95]:
cursor.execute(f'''USE {nombre_bd};''')
query_insert = f'''INSERT INTO {tabla_artista} (id_artista, nombre, oyentes, reproducciones, biografia)
VALUES (%s, %s, %s, %s, %s)'''
df_limpio = df_combinado_artistas.replace({np.nan: None, 'nan': None, 'Nan': None}) #para quitar informacion vacia
df_limpio = df_limpio[['id_artista','nombre','oyentes','reproducciones','biografia']]
df_valores = df_limpio.values.tolist()
cursor.executemany(query_insert, df_valores)
conexion.commit()

In [96]:
cursor.execute(f'''USE {nombre_bd};''')
query_insert = f'''INSERT INTO {tabla_genero_musical} (id_genero, nombre)
VALUES (%s, %s)'''
df_limpio = df_genero_final.replace({np.nan: None, 'nan': None, 'Nan': None}) 
df_limpio = df_limpio[['id','name']]
df_valores = df_limpio.values.tolist()
cursor.executemany(query_insert, df_valores)
conexion.commit()

In [97]:
set(df_albumes["id_artista"]) - set(df_combinado_artistas["id_artista"])

set()

In [98]:
cursor.execute(f'''USE {nombre_bd};''')
query_insert = f'''INSERT INTO {tabla_album} (id_album, id_artista, id_genero, titulo, numero_canciones, fecha_lanzamiento)
VALUES (%s, %s, %s, %s, %s, %s)'''
df_limpio = df_albumes.replace({np.nan: None, 'nan': None, 'Nan': None}) 
df_limpio = df_limpio[['id_album', 'id_artista', 'id_genre', 'titulo', 'n_canciones', 'fecha_lanzamiento']]
df_valores = df_limpio.values.tolist()
cursor.executemany(query_insert, df_valores)
conexion.commit()

In [99]:
cursor.execute(f'''USE {nombre_bd};''')
query_insert = f'''INSERT INTO {tabla_canciones} (id_cancion, id_artista, id_album, titulo, duracion, ranking_lista, colaboraciones)
VALUES (%s, %s, %s, %s, %s, %s, %s)'''

df_limpio = df_canciones.replace({np.nan: None, 'nan': None, 'Nan': None}) #para quitar informacion vacia
df_limpio = df_limpio[['id_cancion', 'id_artista', 'id_album', 'title', 'duration', 'rank', 'contributors_bool']]
df_valores = df_limpio.values.tolist()
cursor.executemany(query_insert, df_valores)
conexion.commit()

In [100]:
diccionario_genero = {"Taylor Swift": "femenino",
                       "Shakira": "femenino", 
                       "Beyoncé": "femenino", 
                       "Rihanna": "femenino", 
                       "Lady Gaga": "femenino", 
                       "Adele": "femenino", 
                       "Madonna": "femenino", 
                       "Britney Spears": "femenino", 
                       "BLACKPINK": "femenino", 
                       "Little Mix":"femenino", 
                       "Coldplay": "masculino",     
                       "Bad Bunny": "masculino",     
                       "Queen": "masculino",     
                       "Eminem": "masculino",     
                       "The Weeknd": "masculino",     
                       "Ed Sheeran": "masculino",     
                       "Metallica": "masculino",     
                       "C. Tangana": "masculino",     
                       "Michael Jackson": "masculino",    
                        "Shawn Mendes": "masculino",     
                        "The Cranberries": "mixto",     
                        "Garbage": "mixto",     
                        "Blondie": "mixto",     
                        "Pretenders": "mixto",     
                        "Amaral": "mixto",     
                        "Roxette": "mixto",     
                        "ABBA": "mixto",     
                        "Evanescence": "mixto",    
                        "Paramore": "mixto", 
                        "La Oreja de Van Gogh": "mixto" }

In [101]:
cursor.execute(f'''USE {nombre_bd};''')
for nombre, genero in diccionario_genero.items(): 
    cursor.execute(f'''UPDATE {tabla_artista} SET genero = %s WHERE nombre = %s''', (genero, nombre))
conexion.commit()

In [103]:
cursor.execute(f'''UPDATE {tabla_artista} SET artista_principal = TRUE WHERE id_artista IN ({','.join(['%s']*len(id_artistas))})''', id_artistas)
cursor.execute(f'''UPDATE {tabla_artista} SET artista_principal = FALSE WHERE id_artista NOT IN ({','.join(['%s']*len(id_artistas))})''', id_artistas)
conexion.commit()

# ANALISIS Y CONSULTAS DE BASE DE DATOS

¿Suena igual para todos? Análisis de la presencia y popularidad femenina en las plataformas de streaming musical
¿Las artistas femeninas tienen la misma representación y éxito que los artistas masculinos en la música digital?

1. ¿Cuál es la media de oyentes y reproducciones de las artistas femeninas frente a los masculinos?
2. Eficiencia por canción: Si dividimos reproducciones entre el número total de canciones, ¿quién obtiene más "rendimiento" por cada tema lanzado?
3. El Techo del Ranking: ¿Cuál es la posición media en ranking_lista para las mujeres frente a los hombres?
4. Presencia en el Top: ¿Qué porcentaje de artistas en el "Top 10" (según ranking_lista) son mujeres?
5. El fenómeno de la colaboración: Según la columna colaboraciones de la tabla canciones, ¿quién colabora más? ¿Las mujeres suelen aparecer más en canciones con colaboraciones que en solitario?
6. Densidad de los álbumes: ¿Quién saca álbumes más largos (más número canciones)?
7. Frecuencia de lanzamiento: Usando fecha_lanzamiento, ¿cuál es el tiempo medio que pasa una mujer entre álbum y álbum frente a los hombres?
8. Géneros "Generizados": ¿En qué estilos musicales (nombre de la tabla genero_musical) hay una presencia nula o mínima de mujeres?


In [ ]:
# 1. ¿Cuál es la media de oyentes y reproducciones de las artistas femeninas frente a los masculinos?

